In [4]:
!pip install sentence-transformers

  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
Using cached click-8.5.0-py3-none-any.whl (125 kB)
  Attempting uninstall: click
    Found existing installation: click 8.2.1
    Uninstalling click-8.2.1:
      Successfully uninstalled click-8.2.1



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
pip install --upgrade pip

  Using cached pip-26.2.1-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.2.1-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 26.0.1
    Uninstalling pip-26.0.1:
      Successfully uninstalled pip-26.0.1
Note: you may need to restart the kernel to use updated packages.


In [9]:
from pathlib import Path
import numpy as np
from sentence_transformers import SentenceTransformer

In [11]:
from pathlib import Path

# Documents list create karo
documents = []

# docs folder ke saare .txt files read karo
for file_path in Path("docs").glob("*.txt"):
    text = file_path.read_text(encoding="utf-8")
    documents.append(text)

print("documents loaded:", len(documents))   

documents loaded: 3


In [13]:
for i,document in enumerate(documents):
    print(f"document{i+1}")
    print('----------')
    print(document)

document1
----------
Deep learning is a subset of machine learning.
It uses artificial neural networks with multiple layers.
Neural networks contain input, hidden and output layers.
CNNs are commonly used for image classification.
RNNs are designed to process sequential data.
document2
----------
Machine learning is a branch of artificial intelligence.
It allows computers to learn patterns from data.
Supervised learning uses labelled training data.
Unsupervised learning finds patterns in unlabelled data.
Common machine learning algorithms include
Linear Regression, Decision Tree and Random Forest.
document3
----------
Python is a high-level programming language.
Python has a simple and readable syntax.
Python is widely used in data science,
web development, automation and artificial intelligence.
Python supports libraries such as NumPy and Pandas.


In [14]:
def split_text(text,chunk_size=20):
    words=text.split()
    chunks=[]
    for i in range(0,len(words),chunk_size):
        chunk=" ".join(
            words[i:i + chunk_size]
        )
        chunks.append(chunk)
    return chunks    

In [15]:
chunks=[]
for document in documents:
    document_chunks=split_text(document)
    chunks.extend(document_chunks)
print('total chunks:',len(chunks))    

total chunks: 7


In [16]:
for i,chunk in enumerate(chunks):
    print(f"chunk{i+1}")
    print("-------------")
    print(chunk)

# iska use thoda iteractive answer dena ke liye hota hai
# ----------------------------------------------------
# for i, chunk in enumerate(chunks):
    
#     print(f"\nChunk {i + 1}")
#     print("File:", chunk["filename"])
#     print("Text:", chunk["text"])

chunk1
-------------
Deep learning is a subset of machine learning. It uses artificial neural networks with multiple layers. Neural networks contain input,
chunk2
-------------
hidden and output layers. CNNs are commonly used for image classification. RNNs are designed to process sequential data.
chunk3
-------------
Machine learning is a branch of artificial intelligence. It allows computers to learn patterns from data. Supervised learning uses labelled
chunk4
-------------
training data. Unsupervised learning finds patterns in unlabelled data. Common machine learning algorithms include Linear Regression, Decision Tree and Random
chunk5
-------------
Forest.
chunk6
-------------
Python is a high-level programming language. Python has a simple and readable syntax. Python is widely used in data science,
chunk7
-------------
web development, automation and artificial intelligence. Python supports libraries such as NumPy and Pandas.


In [18]:
embedding_model=SentenceTransformer(
    'sentence-transformers/all-MiniLM-L6-v2')
print('embedding model loaded successfully!')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

embedding model loaded successfully!


In [19]:
def get_embedding(text):
    embedding=embedding_model.encode(text)
    return embedding

In [20]:
test_embedding=get_embedding(chunks[0])

print('Embedding created successfully.')
print('Number of values:',len(test_embedding))

Embedding created successfully.
Number of values: 384


In [22]:
doc_embeddings=np.array(
    [get_embedding(chunk)
     for chunk in chunks
    ]
)
print('embedding shape:',doc_embeddings.shape)

embedding shape: (7, 384)


In [26]:
def search_documents(question):
    question_embedding=np.array(
        get_embedding(question)
    )
    similarities=(
    doc_embeddings @ question_embedding
    /
    (
        np.linalg.norm(
            doc_embeddings,
            axis=1
        )
        *
        np.linalg.norm(
            question_embedding
        )
    )
    )
    best_index=np.argmax(similarities)

    return(
        chunks[best_index],
        similarities[best_index],
        best_index
    )  
    

In [27]:
question="what is machine learning?"
relevant_chunk,score,index=search_documents(question)

print("question:")
print(question)

print("\n relevant chunk:")
print(relevant_chunk)

print("\n similarity score :")
print(round(float(score),4))

question:
what is machine learning?

 relevant chunk:
Machine learning is a branch of artificial intelligence. It allows computers to learn patterns from data. Supervised learning uses labelled

 similarity score :
0.7892


In [28]:
question='what is python ?'
relevent_chunk,score,index=search_documents(question)

print('question:')
print(question)

print('\n relevent chunk:')
print(relevent_chunk)

print('\n similarity score:')
print(round(float(score),4))

question:
what is python ?

 relevent chunk:
Python is a high-level programming language. Python has a simple and readable syntax. Python is widely used in data science,

 similarity score:
0.802


In [31]:
question='what is deep learning?'
relevent_chunk,score,index=search_documents(question)

print("question:")
print(question)

print("\n relevant chunk:")
print(relevent_chunk)

print("\n similarity score :")
print(round(float(score),4))

question:
what is deep learning?

 relevant chunk:
Deep learning is a subset of machine learning. It uses artificial neural networks with multiple layers. Neural networks contain input,

 similarity score :
0.7779
